# image_fluxdev

**FLUX.2 dev (Full, 32B)** 로 World Reference → 장소 배경 → 인물 초상.
오픈웨이트 포토리얼 1위권 모델(BFL 자체 벤치마크 기준).

| | |
|---|---|
| 커널 | `fluxdev` |
| 출력 | `outputs/image_fluxdev/<시나리오>/` |

---

## 왜 klein 이 아니라 dev(Full) 인가, 그리고 어떻게 GPU 한 장에 넣었나

klein 은 4B 경량 증류판이다. dev(Full)은 32B 풀 모델로 BFL 이 포토리얼
1위 벤치마크로 내세우는 바로 그 모델이지만, 원래 BF16 기준 64GB 가
필요해 이 GPU(48GB) 에 안 들어간다.

**HuggingFace 공식 블로그가 미리 4bit 로 양자화해 통째로 올려 둔
체크포인트(`diffusers/FLUX.2-dev-bnb-4bit`)를 쓴다** — 트랜스포머와
24B Mistral-3 텍스트 인코더 둘 다 이미 양자화돼 있어 필요 VRAM이
~20GB 로 줄어든다. `black-forest-labs/FLUX.2-dev` 게이트 승인이
이미 돼 있어야 한다(ae.safetensors 받을 때 동의 완료).


## 1. 커널 확인


In [ ]:
import sys, importlib

KERNEL = "fluxdev"
NEED   = ["torch", "diffusers", "transformers", "bitsandbytes", "PIL"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.sh --only fluxdev 을 먼저 실행한다."
)

from diffusers import Flux2Pipeline, Flux2Transformer2DModel
print("\nFlux2Pipeline import OK")


## 2. 인증
**필수** — `black-forest-labs/FLUX.2-dev` 게이트 승인이 돼 있어야 한다.


In [ ]:
import os, pathlib
tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    p = pathlib.Path.home() / ".hf_token"
    if p.exists():
        tok = p.read_text().strip()
        os.environ["HF_TOKEN"] = tok

assert tok.startswith("hf_"), (
    "HF_TOKEN 이 없다.\n"
    "  export HF_TOKEN=hf_...   후 커널 재시작\n"
    "  또는  echo hf_... > ~/.hf_token"
)
print("토큰 설정됨:", tok[:6] + "…" + f" ({len(tok)}자)")
print("주의: black-forest-labs/FLUX.2-dev 게이트를 이 계정이 승인했어야 한다.")


## 3. GPU 확인


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU가 없다. 계산 노드에서 실행할 것."
p = torch.cuda.get_device_properties(0)
print(f"GPU  : {p.name}")
print(f"VRAM : {round(p.total_memory / 1e9, 1)} GB")
print(f"CC   : {p.major}.{p.minor}")
print("\n공식 4bit 사전양자화 체크포인트 기준 대략 20GB 필요.")


## 4. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "image_fluxdev"
OUT_DIR = BASE / "outputs" / MODEL

os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]), pathlib.Path(os.environ["TMPDIR"])):
    p.mkdir(parents=True, exist_ok=True)

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개")


## 5. 스타일

In [ ]:
import hashlib

STYLE = ("cohesive painterly realism, muted desaturated palette, "
         "soft directional daylight, film grain, no watermark. if any text, "
         "writing, calligraphy, or a ledger page appears, it must be authentic-"
         "looking traditional Chinese/Hanja characters as used in real Joseon-"
         "era documents — never Korean Hangul, never garbled or illegible "
         "script")

# 장소류(place/extra_place) 이미지를 전부 seed=0으로 뽑았더니 프롬프트가
# 달라도 구도 뼈대가 비슷하고, 심지어 서로 다른 장소끼리 배경이 거의 똑같이
# 나오는 문제가 있었다. place_id 문자열을 해시해서 장소마다 다른 seed를
# 고정 배정한다 (같은 장소는 항상 같은 seed로 재현 가능, 장소끼리는 다른 seed).
def place_seed(pid):
    return int(hashlib.md5(pid.encode()).hexdigest(), 16) % 1_000_000

# 카메라/구도 지시가 없으면 모델이 "단서를 다 프레임에 욱여넣어야 한다"는
# 강박으로 대칭적이고 비현실적인 배치를 만든다. 장소 프롬프트에 공통으로 넣는다.
CAMERA_GUARD = ("eye-level wide establishing shot, single clear camera "
                 "vantage point, natural one-point perspective, physically "
                 "plausible real-world architecture and proportions, "
                 "asymmetrical naturalistic layout — not a staged, mirrored, "
                 "or perfectly symmetrical arrangement.")


## 6. 모델 로드

In [ ]:
import torch
from transformers import Mistral3ForConditionalGeneration
from diffusers import Flux2Pipeline, Flux2Transformer2DModel

REPO_ID = "diffusers/FLUX.2-dev-bnb-4bit"

transformer = Flux2Transformer2DModel.from_pretrained(
    REPO_ID, subfolder="transformer", torch_dtype=torch.bfloat16, device_map="cpu"
)
text_encoder = Mistral3ForConditionalGeneration.from_pretrained(
    REPO_ID, subfolder="text_encoder", dtype=torch.bfloat16, device_map="cpu"
)

pipe = Flux2Pipeline.from_pretrained(
    REPO_ID, transformer=transformer, text_encoder=text_encoder, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()

print("준비됨:", REPO_ID)


## 7. 생성 함수

In [ ]:
STEPS = 50
GUIDANCE = 4.0
SIZE = 1024


def gen(prompt, out_file, seed=0, size=SIZE):
    image = pipe(
        prompt=prompt,
        generator=torch.Generator(device="cuda").manual_seed(seed),
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=size, height=size,
    ).images[0]
    out_file = pathlib.Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    image.save(out_file)
    return out_file


print("준비됨. 다음 셀(연결 확인)에서 한 장만 먼저 뽑아볼 것.")


## 8. 확장 프롬프트(단서 아이콘 + 전신)

In [ ]:
# ---------------------------------------------------------------------------
# 78장 스펙 확장 이미지 파일럿 — 12_흥부전 하나만.
# 1) 단서 아이콘: clue_graph 에서 "물건/문서/장부"류(S1~S5, 실물 증거)만 뽑는다.
#    증언/소문/검안(K*) 은 사람 말·서류 판독이지 실물이 아니라서 아이콘화하지
#    않는다 — 필요하면 UI 쪽에서 범용 "증언 카드" 아이콘을 재사용하는 게 맞다.
# 2) 인물 전신: 기존 appearance 데이터 그대로, 프레이밍만 waist-up -> full-body.
# ---------------------------------------------------------------------------
import json as _json

PILOT_ID = "12_heungbu"
d = DATA[PILOT_ID]
era, loc = d["setting"]["era"], d["setting"]["location"]

raw = _json.load(open(IN_DIR / "json" / "12_흥부전.json", encoding="utf-8"))
clue_graph = raw["clue_graph"]
PHYSICAL_CLUE_MEDIA = {"물건", "문서", "장부"}
physical_clues = [c for c in clue_graph if c.get("medium") in PHYSICAL_CLUE_MEDIA]

# 5개뿐이라 손으로 번역한다(기계번역 없이, appearance/clues 때와 같은 이유).
CLUE_EN = {
    "S1": "a cloth sack of grain, brought from a poor woodcutter's thatched "
          "cottage — the kind of grain he once received from his wealthier "
          "brother's storehouse in leaner days",
    "S2": "a folded ledger page from a reception-room cabinet, recording sums "
          "quietly funneled away to relatives",
    "S3": "a double ledger book from a steward's room, its second set of "
          "figures showing storehouse goods skimmed off to cover gambling debts",
    "S4": "a few cut logs of timber stacked at a laborer's house, freshly cut "
          "and clearly taken without permission from someone else's mountain land",
    "S5": "a small paper packet of dangerous medicine, folded shut, the kind "
          "quietly prepared and sold outside proper channels",
}

def clue_icon_prompt(cid):
    return (f"a single isolated object, {CLUE_EN[cid]}. centered on a plain "
            f"neutral studio-grey background, product-icon style, no people, "
            f"no hands, no text, no watermark, no legible writing, blank "
            f"pages if a book or ledger, no other objects in frame, no "
            f"birds, no swallows. period: {era}. soft even studio lighting. "
            f"{STYLE}.")

def fullbody_prompt(c):
    # hidream_o1 이 "neutral plain background" 지시를 무시하고 직업에 맞는
    # 소품(바구니, 농기구 등)과 배경 환경을 마음대로 채워 넣는 게 실측으로
    # 확인됐다 — fluxdev 는 안 그랬다. 반복·강조해서 밀어붙인다.
    return (f"full-body portrait of a {c['age']}-year-old {c['gender']}, "
            f"{c['occupation']}, period: {era}. {c['appearance']} "
            f"standing fully upright, facing the camera directly, "
            f"symmetrical frontal pose, arms relaxed at sides, hands empty, "
            f"head to toe fully visible. completely plain empty studio "
            f"background, no props, no objects, no tools, no environment, "
            f"no location, seamless neutral backdrop, even studio lighting. "
            f"{STYLE}.")

print(f"단서 아이콘 {len(physical_clues)}개:", [c['id'] for c in physical_clues])
print(f"전신 일러스트 {len(d['cast'])}개:", [c['id'] for c in d['cast']])


## 9. 생성

In [ ]:
EXT_OUT = OUT_DIR / PILOT_ID

for c in physical_clues:
    cid = c["id"]
    out_f = EXT_OUT / "clue_icons" / f"{cid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", cid); continue
    gen(clue_icon_prompt(cid), out_f, seed=0)
    print("OK clue_icon", cid)

for c in d["cast"]:
    if c["id"] == "victim":
        continue  # 전신은 "주인 있는 장소만" - 피해자는 소유 장소가 없다
    out_f = EXT_OUT / "fullbody" / f"{c['id']}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", c['id']); continue
    gen(fullbody_prompt(c), out_f, seed=0, size=768)  # 세로 프레이밍에 가깝게
    print("OK fullbody", c['id'])

print("\n저장 위치:", EXT_OUT)


## 10. 표정 변형

In [ ]:
# ---------------------------------------------------------------------------
# 표정 3단 변형 (평온 / 흔들림 / 무너짐) — persona.personality/voice_hint 를
# 참고해서 인물별 어조에 맞는 표정 묘사를 쓴다. appearance 핵심 문장은 그대로
# 유지해서(얼굴 특징 묘사 안 바꿈) 세 장이 같은 사람처럼 보이게 한다.
# ---------------------------------------------------------------------------
EMOTION_STATES = {
    "calm":    "calm, composed neutral expression, steady eyes, relaxed posture",
    "shaken":  "visibly uneasy expression, eyes darting slightly, tense shoulders, "
               "a flicker of hesitation crossing the face",
    "breaking":"expression cracking under pressure, eyes wide or downcast, "
               "barely holding composure, visible distress",
}

def emotion_portrait_prompt(c, state_desc):
    return (f"portrait of a {c['age']}-year-old {c['gender']}, {c['occupation']}. "
            f"{c['appearance']} {state_desc}. "
            f"setting: {era}. waist-up framing, neutral background, even lighting. "
            f"{STYLE}.")


## 11. 표정 생성

In [ ]:
for c in d["cast"]:
    if c["id"] == "victim":
        continue  # 피해자는 표정 3단 대신 실루엣 1장 (아래 별도 처리)
    for state, desc in EMOTION_STATES.items():
        out_f = EXT_OUT / "portraits_emotion" / f"{c['id']}_{state}.png"
        if out_f.exists():
            print("   (이미 있음, 건너뜀)", c['id'], state); continue
        gen(emotion_portrait_prompt(c, desc), out_f, seed=0)
        print("OK emotion", c['id'], state)

print("\n저장 위치:", EXT_OUT / "portraits_emotion")


## 12. 2차 확장 프롬프트 (스토리컷/주검그림/현장전경/신규장소/단서전체)

In [ ]:
# ---------------------------------------------------------------------------
# 78(72)장 스펙 2차 확장 — 스토리컷 10 + 주검그림 1 + 현장전경 1 + 신규장소 2
# (우물가/마방) + 단서 썸네일 18개 전부(clue_graph 16 + 신규장소분 XPX1/XPX2).
# 전부 손 번역 — json 원본(death.scene_description, clue_graph[].surface)과
# 팀 문서(KakaoTalk 스크린플레이)의 한글 나레이션을 영어로 옮긴 것.
# ---------------------------------------------------------------------------

# ---- 스토리 컷 10장 (S005~S014, 팀 문서 나레이션 그대로 번역) --------------
palette = d["adapted_story"]["palette"]
tone = d["adapted_story"]["tone"]

# 4번째 필드: 이 컷에 등장하는 인물의 cast id (없으면 None) — appearance
# 텍스트를 주입해서 다른 카테고리(초상/전신)의 그 인물과 같아 보이게 한다.
# 원작 인물 <-> 우리 배역 대응: 흥부=C1, 놀부=victim(피해자), 놀부 처=C2,
# 마름(집사)=C3, 이웃 일꾼=C4, 약초 노파=C5.
# 5번째 필드: 이 컷에서 제비를 보여줘도 되는지. 원작 나레이션 특성상 흥부전
# 자체가 "제비/박"과 워낙 강하게 엮여 있어서, 프롬프트에 안 써도 모델이
# 알아서 새를 그려 넣는 게 실측으로 확인됐다(주검그림에서도 발생) — 제비가
# 실제로 필요한 컷(02, 03) 말고는 명시적으로 금지한다.
STORY_CUTS = [
    ("cut01", "Long ago, there lived a kind-hearted younger brother, Heungbu, "
               "and his wealthy elder brother, Nolbu.",
     "a poor thatched cottage and a grand tiled estate standing side by side "
     "in a Korean village, establishing shot", None, False),
    ("cut02", "Driven out and left poor by his brother, Heungbu mended the "
               "broken leg of a swallow, and in gratitude it brought him a "
               "gourd seed that grew into gold and treasure, making him rich.",
     "Heungbu kneeling by his cottage eaves, gently bandaging an injured "
     "swallow's leg with cloth", "C1", True),
    ("cut03", "Envious of his brother's fortune, Nolbu deliberately broke a "
               "swallow's leg himself, meaning to mend it and grow his own "
               "gourd of treasure — but the scheme brought him only ruin and "
               "humiliation.",
     "Nolbu crouched in his courtyard, roughly handling a swallow with a "
     "guilty, calculating expression", "victim", True),
    ("cut04", "His storehouses were full of grain, yet he wouldn't spare a "
               "beggar at his gate so much as a sip of water — his bitter "
               "stinginess left many people around him quietly suffering.",
     "Nolbu standing protectively in front of overflowing storehouse chests, "
     "arms crossed, a beggar turned away at the gate in the distance", "victim", False),
    ("cut05", "Nolbu's wife had grown weary of her husband's stinginess, and "
               "had quietly been funneling household wealth to her own family.",
     "Nolbu's wife alone in her room, carefully wrapping coins and valuables "
     "into a cloth bundle, glancing at the door", "C2", False),
    ("cut06", "The steward, entrusted with the storehouse, the household's "
               "wealth, and even the medicinal tonic brewed for Nolbu every "
               "night, always bowed and scraped outwardly — while secretly "
               "harboring designs of his own.",
     "the steward bowing low with an obsequious smile, one hand hidden "
     "behind his back near the storehouse ledger", "C3", False),
    ("cut07", "The neighboring laborer, who tended Nolbu's mountain land, "
               "had been cheated out of his wages more than once and nursed "
               "a deep resentment.",
     "a sun-browned laborer standing in a mountain field, jaw clenched, "
     "fists tight at his sides", "C4", False),
    ("cut08", "The old herbalist who prepared Nolbu's medicine had once, "
               "through a small illicit substitution, caused his face to "
               "swell.",
     "an elderly herbalist grinding herbs at a low table, a small "
     "unmarked packet of medicine tucked just out of sight", "C5", False),
    ("cut09", "That night, fearing another attack, Nolbu made his usual "
               "preparations and took his medicinal tonic in the reception "
               "room, just as he did every night.",
     "Nolbu seated at his low desk in the reception room, lifting a bowl of "
     "medicinal tonic to drink, lamplight warm around him", "victim", False),
    ("cut10", "But this night, Nolbu was found collapsed beside the "
               "overturned tonic bowl.",
     "an overturned medicinal bowl on the reception room floor, dark "
     "reddish residue spreading across the boards, the room otherwise "
     "undisturbed", None, False),
]

def story_cut_prompt(narration, scene_desc, char_id, birds_ok):
    char_text = ""
    if char_id:
        person = next(c for c in d["cast"] if c["id"] == char_id)
        char_text = f" character appearance: {person['appearance']}"
    bird_guard = "" if birds_ok else " no birds, no swallows, no flying birds."
    return (f"{scene_desc}.{char_text} "
            f"story context: {narration} "
            f"setting: {era}, {loc}. mood: {d['source_story']['tone']}. "
            f"color palette: {palette}.{bird_guard} {STYLE}.")

# ---- 주검그림 1장 (피해자 발견 장면, 유일하게 사람/시신을 보여주는 컷) ------
BODY_IMAGE_KO = raw["death"]["scene_description"]
# "gourd"(박) 단어를 프롬프트에 쓰면 흥부전 원작의 제비/박 상징이 너무 강해서
# 모델이 살인 현장에 제비·비둘기·박 열매를 자동으로 그려 넣는다(실측으로
# fluxdev/hidream_o1 둘 다 확인됨, hidream_o1 은 박 열매까지 그렸다). 그래서
# "박 깨는 톱" 대신 그냥 "목공용 톱"으로 바꾸고 gourd 라는 단어 자체를 뺐다.
# 장부도 "펼쳐진 문서"가 아니라 "덮인 채 안 보이는 책"으로 바꿔서 가짜 글씨가
# 그려질 여지를 줄인다.
BODY_IMAGE_EN = ("Nolbu's reception room. A dark reddish residue spreads from "
                  "an overturned bowl of medicinal tonic. Nolbu lies fallen, "
                  "clutching a closed cloth-bound book to his chest, its "
                  "pages shut and not visible. A hand-saw for woodworking "
                  "rests set aside nearby.")

def body_image_prompt():
    return (f"a somber discovery scene, viewed from a respectful middle "
            f"distance, figure partially obscured by furniture and shadow "
            f"— dignified, not graphic. exactly one person, fallen, no "
            f"other figures, no bystanders, nobody else standing or "
            f"present anywhere in the frame. {BODY_IMAGE_EN} "
            f"setting: {era}, {loc}. color palette: {palette}. "
            f"no birds, no swallows, no flying birds, no whimsical "
            f"fairy-tale elements. no legible writing, no visible text or "
            f"characters on any book or page. {STYLE}.")

# ---- 현장 전경 1장 (범행 장소 와이드 샷, 사람 없이) ------------------------
def scene_overview_prompt():
    pc = next(p for p in d["places"] if p["place_id"] == "PC")
    return (f"a wide establishing view of the crime scene, empty of people, "
            f"no bodies, no corpse. {pc['name']}. {pc['description']} "
            f"this scene must clearly show: {'; '.join(pc['clues'])}. "
            f"setting: {era}, {loc}. color palette: {palette}. "
            f"no birds, no swallows, no flying birds, no whimsical "
            f"fairy-tale elements. "
            f"completely vacant, deserted. {STYLE}.")

# ---- 신규 장소 2곳 (우물가/마방, clue_graph에 XPX1/XPX2로 연결) -----------
EXTRA_PLACES = {
    "PX1": {
        "name": "The Well",
        "description": ("A village well with a well-sweep and a wooden water "
                         "bucket, a stone rim worn smooth, a water barrel "
                         "resting nearby."),
        "clues": ["a patch of ground near the well still damp, though "
                   "everything else has long since dried"],
    },
    "PX2": {
        "name": "The Stable",
        "description": ("A modest stable housing a donkey, with a peddler's "
                         "pack and goods stacked in the corner, straw "
                         "scattered on the floor."),
        "clues": ["a fresh set of hoofprints and cart tracks leading out "
                   "toward the mountain path, still crisp in the dirt"],
    },
}

def extra_place_prompt(pid):
    p = EXTRA_PLACES[pid]
    clue_list = "; ".join(p["clues"])
    return (f"empty, uninhabited space — no people, no humans, no figures, "
            f"no bodies, no corpse, nobody present anywhere in frame. "
            f"{p['name']}. {p['description']} {CAMERA_GUARD} "
            f"this scene must clearly show: {clue_list} — as objects and "
            f"traces only, with no person or body present to go with them. "
            f"period and place: {era}, {loc}. mood: {tone}. "
            f"color palette: {palette}. no birds, no swallows, no flying "
            f"birds, no whimsical fairy-tale elements. "
            f"completely vacant, deserted, unoccupied, empty of any person "
            f"or body no matter what else is implied. {STYLE}.")

# ---- 단서 썸네일 18장 (clue_graph 16개 전부 + XPX1/XPX2) -------------------
# medium 별로 그릴 대상이 다르다: 실물(물건/문서/장부)은 그 물건 자체를,
# 증언/소문/검안/기록/현장은 "그 정보가 적힌/드러난 매개체"를 아이콘화한다
# (증언 카드, 검안 기록지, 소문이 적힌 쪽지 등 — 사람 얼굴은 절대 안 보여준다,
# 이 라운드 안에서 범인이 특정되면 안 되므로).
CLUE_EN_FULL = {
    "K1": "an autopsy record scroll, brush-written notes describing "
          "cyanosis and arsenic residue found in a stomach examination",
    "K2": "a medicine bowl beside a low desk, positioned as though someone "
          "simply collapsed from long illness — deceptively ordinary",
    "K3": "a folded testimony note, an elderly herbalist's account written "
          "out, describing a night spent together at a hut",
    "K4": "a folded testimony note, a laborer's account written out, "
          "describing a night spent home with his wife",
    "K5": "a folded testimony note, an apprentice's account written out, "
          "corroborating a night spent drying herbs at a hut",
    "K6": "an open household ledger, a suspicious entry circled where "
          "storehouse goods were quietly skimmed off",
    "K7": "a folded note recording overheard rumor, describing a heated "
          "argument witnessed between two people",
    "K8": "a forensic diagram sketch, showing wound angle and height "
          "annotations from an examination",
    "K9": "a diagram of footprints found near the scene, evenly spaced "
          "marks annotated on a floor plan",
    "K10": "a folded testimony note corroborating another witness's account, "
           "placing someone at a hut that night",
    "K11": "a folded testimony note corroborating another witness's "
           "account, placing someone at home that night",
    "S1": "a cloth sack of grain, brought from a poor woodcutter's thatched "
          "cottage — the kind of grain he once received from his wealthier "
          "brother's storehouse in leaner days",
    "S2": "a folded ledger page from a reception-room cabinet, recording "
          "sums quietly funneled away to relatives",
    "S3": "a double ledger book from a steward's room, its second set of "
          "figures showing storehouse goods skimmed off to cover gambling "
          "debts",
    "S4": "a few cut logs of timber stacked at a laborer's house, freshly "
          "cut and clearly taken without permission from someone else's "
          "mountain land",
    "S5": "a small paper packet of dangerous medicine, folded shut, the "
          "kind quietly prepared and sold outside proper channels",
    "XPX1": "a patch of damp ground near a village well, water darkening "
            "the soil though everything nearby has long since dried",
    "XPX2": "fresh hoofprints and cart-wheel tracks in a stable's dirt "
            "floor, leading out toward a mountain path",
}

def clue_icon_prompt_full(cid):
    return (f"a single isolated object, {CLUE_EN_FULL[cid]}. centered on a "
            f"plain neutral studio-grey background, product-icon style, no "
            f"people, no hands, no faces, no text, no watermark, no legible "
            f"writing, blank pages if a book, scroll, ledger or note "
            f"appears, no birds, no swallows, no other objects in frame. "
            f"period: {era}. soft even studio lighting. {STYLE}.")

print("스토리컷:", len(STORY_CUTS))
print("단서 전체:", len(CLUE_EN_FULL))
print("신규 장소:", list(EXTRA_PLACES.keys()))


## 13. 2차 확장 생성

In [ ]:
EXT_OUT = OUT_DIR / PILOT_ID

# 스토리 컷
for cid, narration, scene, char_id, birds_ok in STORY_CUTS:
    out_f = EXT_OUT / "story_cuts" / f"{cid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", cid); continue
    gen(story_cut_prompt(narration, scene, char_id, birds_ok), out_f, seed=0)
    print("OK story_cut", cid)

# 주검그림
out_f = EXT_OUT / "body_image.png"
if not out_f.exists():
    gen(body_image_prompt(), out_f, seed=0)
    print("OK body_image")
else:
    print("   (이미 있음, 건너뜀) body_image")

# 현장 전경
out_f = EXT_OUT / "scene_overview.png"
if not out_f.exists():
    gen(scene_overview_prompt(), out_f, seed=0)
    print("OK scene_overview")
else:
    print("   (이미 있음, 건너뜀) scene_overview")

# 신규 장소 2곳 (장소배경 + 지도썸네일용 원본) — 장소마다 다른 seed
for pid in EXTRA_PLACES:
    out_f = EXT_OUT / "places" / f"{pid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", pid); continue
    gen(extra_place_prompt(pid), out_f, seed=place_seed(pid))
    print("OK extra_place", pid)

# 단서 썸네일 18장 전부 (기존 clue_icons 폴더에 이어서 채운다)
for cid in CLUE_EN_FULL:
    out_f = EXT_OUT / "clue_icons" / f"{cid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", cid); continue
    gen(clue_icon_prompt_full(cid), out_f, seed=0)
    print("OK clue_icon", cid)

print("\n저장 위치:", EXT_OUT)


## 14. 피해자 실루엣

In [ ]:
# ---------------------------------------------------------------------------
# 피해자 실루엣 1장 — 문서 스펙: "얼굴이 보이지 않게 어두운 실루엣.
# 시선 방향이 카메라 쪽을 향하되 위협적이지 않게." 용의자용 표정 3단과는
# 완전히 다른 별도 처리.
# ---------------------------------------------------------------------------
def victim_silhouette_prompt():
    victim = next(c for c in d["cast"] if c["id"] == "victim")
    return (f"a dark silhouette portrait of a {victim['age']}-year-old "
            f"{victim['gender']} figure, facing toward the camera but not "
            f"threatening, face entirely obscured in shadow, no facial "
            f"features visible. {victim['occupation']}. "
            f"setting: {era}. waist-up framing, dim moody backlighting, "
            f"neutral dark background. {STYLE}.")


## 15. 실루엣 생성

In [ ]:
out_f = EXT_OUT / "portraits_emotion" / "victim_silhouette.png"
if not out_f.exists():
    gen(victim_silhouette_prompt(), out_f, seed=0)
    print("OK victim_silhouette")
else:
    print("   (이미 있음, 건너뜀) victim_silhouette")


## 16. 재연 컷 설정 (텍스트 일관성)

In [ ]:
# ---------------------------------------------------------------------------
# 재연 컷 5장 — Qwen-Image-Edit(image+text)은 화질 열화로 제외.
# 대신 범인(C3)의 appearance 텍스트를 다른 초상들과 똑같이 재사용해서
# 텍스트만으로 일관성을 맞춘다 (이 파이프라인 전체가 쓰는 방식과 동일).
# ---------------------------------------------------------------------------
CULPRIT_ID = "C3"
culprit = next(c for c in d["cast"] if c["id"] == CULPRIT_ID)

REENACT_CUTS = [
    ("cut1_that_night",
     "his face hardening with resolve in dim lamplight — if the truth "
     "about the storehouse ledger comes out, he will be turned over to "
     "the authorities. close-up, night setting, tense expression."),
    ("cut2_approach",
     "seen from behind, walking quietly down a dim hallway toward a "
     "reception room at night, unnoticed by anyone, tense posture, "
     "candle-lit corridor."),
    ("cut3_the_act",
     "a dim reception room at night, reaching toward a bowl of medicinal "
     "tonic on a low desk, implied tension rather than graphic action, "
     "dark and shadowy composition."),
    ("cut4_cover_up",
     "face contorting into feigned anguish and grief, false tears, "
     "performing distress for onlookers, dim interior setting."),
    ("cut5_next_morning",
     "standing calm and composed in daylight outside a quiet reception "
     "room at dawn, deceptively serene expression, soft morning light, "
     "no one else around."),
]

def reenact_prompt(desc):
    return (f"a {culprit['age']}-year-old {culprit['gender']}, "
            f"{culprit['occupation']}. {culprit['appearance']} {desc} "
            f"setting: {era}, {loc}. mood: tense, quietly dreadful. "
            f"color palette: {palette}. no birds, no swallows, no "
            f"whimsical fairy-tale elements. {STYLE}.")

print(f"재연 컷 {len(REENACT_CUTS)}개, 범인: {CULPRIT_ID}")


## 17. 재연 컷 생성

In [ ]:
for cid, desc in REENACT_CUTS:
    out_f = EXT_OUT / "reenact_cuts" / f"{cid}.png"
    if out_f.exists():
        print("   (이미 있음, 건너뜀)", cid); continue
    gen(reenact_prompt(desc), out_f, seed=0)
    print("OK reenact", cid)

print("\n저장 위치:", EXT_OUT / "reenact_cuts")
